Libraries

In [1]:
import yfinance as yf
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

Initial setup

In [2]:
import numpy as np

tickers = pd.read_csv(
    "asx100_names.csv"
)["tickers"].tolist()
tickers.pop()
print(tickers)

data = yf.download(
    tickers,
    start="2015-01-01",
    auto_adjust=True,
    group_by="ticker",
    threads=True
)

# build master data panel
dfs = []
for ticker in tickers:
    try:
        temp = data[ticker].copy()
        temp["Ticker"] = ticker
        temp = temp.reset_index()
        dfs.append(temp)
    except:
        print(f"Missing {ticker}")

panel = pd.concat(dfs)

['A2M.AX', 'AFI.AX', 'AGL.AX', 'AIA.AX', 'ALD.AX', 'ALL.AX', 'ALQ.AX', 'ALX.AX', 'AMC.AX', 'ANN.AX', 'ANZ.AX', 'APA.AX', 'APT.AX', 'ARG.AX', 'AST.AX', 'ASX.AX', 'AWC.AX', 'AZJ.AX', 'BEN.AX', 'BHP.AX', 'BLD.AX', 'BOQ.AX', 'BSL.AX', 'BXB.AX', 'CBA.AX', 'CCL.AX', 'CHC.AX', 'CIM.AX', 'COH.AX', 'COL.AX', 'CPU.AX', 'CSL.AX', 'CWN.AX', 'CWY.AX', 'DMP.AX', 'DXS.AX', 'EVN.AX', 'FBU.AX', 'FMG.AX', 'FPH.AX', 'GMG.AX', 'GPT.AX', 'HVN.AX', 'IAG.AX', 'IEL.AX', 'IGO.AX', 'IPL.AX', 'JBH.AX', 'JHX.AX', 'LLC.AX', 'MCY.AX', 'MEZ.AX', 'MFG.AX', 'MGOC.AX', 'MGR.AX', 'MIN.AX', 'MPL.AX', 'MQG.AX', 'NAB.AX', 'NCM.AX', 'NST.AX', 'NXT.AX', 'ORG.AX', 'ORI.AX', 'OSH.AX', 'OZL.AX', 'PMGOLD.AX', 'QAN.AX', 'QBE.AX', 'QUB.AX', 'REA.AX', 'REH.AX', 'RHC.AX', 'RIO.AX', 'RMD.AX', 'S32.AX', 'SCG.AX', 'SEK.AX', 'SGP.AX', 'SHL.AX', 'SOL.AX', 'SPK.AX', 'STO.AX', 'SUN.AX', 'SVW.AX', 'SYD.AX', 'TAH.AX', 'TCL.AX', 'TLS.AX', 'TPG.AX', 'TWE.AX', 'VAS.AX', 'VCX.AX', 'WBC.AX', 'WES.AX', 'WOR.AX', 'WOW.AX', 'WPL.AX', 'WTC.AX', 'XRO.

[***********           23%                       ]  23 of 100 completed$SYD.AX: possibly delisted; no timezone found
[************          24%                       ]  24 of 100 completed$CIM.AX: possibly delisted; no timezone found
[****************      34%                       ]  34 of 100 completed$BLD.AX: possibly delisted; no timezone found
[*****************     36%                       ]  36 of 100 completed$OSH.AX: possibly delisted; no timezone found
[*****************     36%                       ]  36 of 100 completed$NCM.AX: possibly delisted; no timezone found
$CWN.AX: possibly delisted; no timezone found
[******************    38%                       ]  38 of 100 completed$IPL.AX: possibly delisted; no timezone found
[**********************47%                       ]  47 of 100 completed$OZL.AX: possibly delisted; no timezone found
[**********************62%*****                  ]  62 of 100 completed$WPL.AX: possibly delisted; no timezone found
[*****************

Feature Engineering:

In [3]:
#Momentum and reversal features
panel["mom_1m"] = panel.groupby("Ticker")["Close"].pct_change(21)
panel["mom_3m"] = panel.groupby("Ticker")["Close"].pct_change(63)
panel["mom_6m"] = panel.groupby("Ticker")["Close"].pct_change(126)
panel["mom_12m"] = panel.groupby("Ticker")["Close"].pct_change(252)
panel["reversal_1m"] = -panel["mom_1m"]

#Daily Returns
panel["daily_return"] = panel.groupby("Ticker")["Close"].pct_change()

# Add market returns
market = yf.download("^AXJO", start="2015-01-01", auto_adjust=True)
market.columns = market.columns.get_level_values(0)
market = market.reset_index()
market["market_return"] = market["Close"].pct_change()
panel = panel.merge(
    market[["Date", "market_return"]],
    on="Date",
    how="left"
)
print(panel[["Date", "Ticker", "market_return"]].head())

# Volatility features (annualized)
panel["vol_20"] = (
    panel.groupby("Ticker")["daily_return"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
) * np.sqrt(252)

panel["vol_60"] = (
    panel.groupby("Ticker")["daily_return"]
    .rolling(60)
    .std()
    .reset_index(level=0, drop=True)
) * np.sqrt(252)

#Ratio: this is short term to long term volatility
panel["vol_ratio"] = (panel["vol_20"] /panel["vol_60"])

# downside volatility
panel["negative_return"] = panel["daily_return"].clip(upper=0)
panel["downside_vol"] = (
    panel.groupby("Ticker")["negative_return"]
    .transform(lambda x: x.rolling(60).std())
) * np.sqrt(252)


#Liquidity
panel["dollar_volume"] = (panel["Close"] * panel["Volume"])

avg_volume = (
    panel.groupby("Ticker")["Volume"]
    .rolling(20)
    .mean()
    .reset_index(level=0, drop=True)
)

panel["volume_shock"] = (panel["Volume"]/avg_volume)

# Illiquidity
panel["amihud"] = (
    panel["daily_return"].abs()/ panel["dollar_volume"]
)

# Beta features (short, medium and long term)
def calculate_beta(group, window):
    covariance = (
        group["daily_return"]
        .rolling(window)
        .cov(group["market_return"])
    )
    market_variance = (
        group["market_return"]
        .rolling(window)
        .var()
    )
    return covariance / market_variance

panel["beta_252"] = (
    panel.groupby("Ticker", group_keys=False)
    .apply(calculate_beta, window=252)
)

panel["beta_126"] = (
    panel.groupby("Ticker", group_keys=False)
    .apply(calculate_beta, window=126)
)  

panel["beta_63"] = (
    panel.groupby("Ticker", group_keys=False)
    .apply(calculate_beta, window=63)
)

# risk features
def calculate_idio_volatility(residualwindow):
    panel[f"expected_return_{residualwindow}"] = (
        panel[f"beta_{residualwindow}"] * panel["market_return"]
    )
    panel[f"residual_return_{residualwindow}"] = (
        panel["daily_return"] - panel[f"expected_return_{residualwindow}"]
    )
    return (
        panel.groupby("Ticker")[f"residual_return_{residualwindow}"]
        .rolling(60)
        .std()
        .reset_index(level=0, drop=True)
    )
panel["idio_vol_252"] = calculate_idio_volatility(252)
panel["idio_vol_126"] = calculate_idio_volatility(126)
panel["idio_vol_63"] = calculate_idio_volatility(63)

print("Before cleaning:", panel.shape)
#panel = panel.dropna()
#print("after cleaning", panel.shape )

# TO ADD:
# - Market regime variables, macro variables
# - interaction variables between features (e.g., momentum * volatility, etc.)

# cross-sectional standardization:
feature_cols = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "mom_12m",
    "reversal_1m",
    "vol_20",
    "vol_60",
    "vol_ratio",
    "downside_vol",
    "volume_shock",
    "dollar_volume", #double check
    "amihud",
    "beta_252",
    "beta_126",
    "beta_63",
    "idio_vol_252",
    "idio_vol_126",
    "idio_vol_63"
]

for feature in feature_cols:
    panel[f"{feature}_z"] = (
        panel.groupby("Date")[feature]
        .transform(
            lambda x:
            (x - x.mean()) / x.std()
        )
    )
    feature = f"{feature}_z"  

# targets - excess return over next 1 month (vs market return)
panel = panel.sort_values(["Ticker", "Date"])
panel["future_return_1m"] = (
    panel.groupby("Ticker")["Close"]
    .shift(-21)
    .div(panel["Close"])
    .sub(1)
)
# future market return
panel["future_market_return_1m"] = (
    panel.groupby("Ticker")["market_return"].transform(
        lambda x: (
            (1 + x)
            .rolling(21)
            .apply(np.prod, raw=True)
            .shift(-20)
            - 1
        )
    )
)
panel["future_excess_return"] = (panel["future_return_1m"]- panel["future_market_return_1m"])
# for lightgbm, the target variables are quantile bins 
panel["target_relevance"] = (
    panel.groupby("Date")["future_excess_return"]
    .transform(
        lambda x: pd.qcut(
            x,
            q=5,
            labels=False,
            duplicates="drop"
        )
    )
)

[*********************100%***********************]  1 of 1 completed


Price       Date  Ticker  market_return
0     2015-01-02  A2M.AX            NaN
1     2015-01-05  A2M.AX       0.002649
2     2015-01-06  A2M.AX      -0.015687
3     2015-01-07  A2M.AX      -0.002088
4     2015-01-08  A2M.AX       0.005211
Before cleaning: (292600, 35)


c:\Users\tiany\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Train/test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb
from lightgbm import LGBMRanker
from scipy.stats import spearmanr

model_data = panel.dropna(subset=feature_cols + ["target_relevance"])
train = model_data[model_data["Date"] < "2022-01-01"]
validate = model_data[(model_data["Date"] >= "2022-01-01") & (model_data["Date"] < "2024-01-01")]
test = model_data[model_data["Date"] >= "2024-01-01"]
X_train = train[feature_cols]
X_valid = validate[feature_cols]
X_test = test[feature_cols]
y_train = train["target_relevance"]
y_valid = validate["target_relevance"]
y_test = test["target_relevance"]

train_group = (
    train
    .groupby("Date")
    .size()
    .to_numpy()
)

valid_group = (
    validate
    .groupby("Date")
    .size()
    .to_numpy()
)

ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    learning_rate=0.05,
    n_estimators=300,
    num_leaves=31,
    random_state=42
)

ranker.fit(
    X_train,
    y_train,
    group=train_group,
    eval_set=[(X_valid, y_valid)],
    eval_group=[valid_group],
    eval_at=[10]
)

test["lgb_prediction"] = ranker.predict(X_test)

test["predicted_rank"] = (
    test
    .groupby("Date")
    ["lgb_prediction"]
    .rank(
        ascending=False,
        pct=True
    )
)
# Test feature importance
importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": ranker.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)
print(importance)

# Validation metric IC: information coefficient for quantitative investing
# correlation with actual rankings: +1 to -1
# basically a correlation coefficient but suitable for quantittative models and investing
ic = (
    test
    .groupby("Date")
    .apply(
        lambda x:
        spearmanr(
            x["lgb_prediction"],
            x["future_excess_return"]
            )[0]
    )
)
print(test["lgb_prediction"].describe())
ic = ic.dropna()
print(ic.mean())
print(ic.describe())

test["inverse_prediction"] = -test["lgb_prediction"]

#inverse check
ic_inverse = (
    test.groupby("Date")
    .apply(
        lambda x: spearmanr(
            x["inverse_prediction"],
            x["future_excess_return"]
        )[0]
    )
)
print(ic_inverse.mean())

# check quantiles
panel.groupby("target_relevance")["future_excess_return"].mean()


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001401 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 45939, number of used features: 18
          Feature  Importance
2          mom_6m         852
12       beta_252         846
3         mom_12m         821
8    downside_vol         734
13       beta_126         718
14        beta_63         636
7       vol_ratio         613
1          mom_3m         581
5          vol_20         568
6          vol_60         518
10  dollar_volume         363
15   idio_vol_252         358
17    idio_vol_63         291
0          mom_1m         279
4     reversal_1m         276
16   idio_vol_126         266
9    volume_shock         177
11         amihud         103
count    53013.000000
mean        -0.553381
std          1.261625
min         -5.013622
25%         -1.497457
50%         -0.571219
75%      